# Structured Exercise: Receptive Field and Pooling in Image Reconstruction

The goal of this exercise is to understand, through a small PyTorch experiment, why the receptive field of a convolutional network matters in computational imaging. This idea will become important later, when deeper architectures and `UNet`-type models are introduced.

Consider the following setting. Start from a clean grayscale image $\mathbf{x}$, create a blurred version $\mathbf{y} = K\mathbf{x}$, and train a small CNN to predict $\mathbf{x}$ from $\mathbf{y}$. Compare two networks:

*   **Model A**: a shallow CNN with a few `Conv2d` layers and no pooling;
*   **Model B**: a CNN that includes at least one downsampling step (for example `MaxPool2d` or a stride-2 convolution) and one upsampling step.

The purpose is not to obtain the best reconstruction quality, but to observe how enlarging the effective receptive field changes the result.

## Task 1: Build the dataset

Use the ideas already shown in this notebook.

1.  Start from one or more clean images available in the course material.
2.  Build a small training subset with a dataloader, for example by selecting a limited number of Mayo images.
3.  Corrupt them with a blur operator. You may use the IPPy blurring operator or a standard `Conv2d` with fixed kernel.
4.  Keep a small validation set aside.

In [1]:
import glob
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

here = Path.cwd().resolve()
for base in (here, *here.parents):
    if (base / 'Mayo').exists():
        book_root = base
        break
else:
    raise FileNotFoundError('Could not locate the course root containing Mayo.')

for base in (here, *here.parents):
    if (base / 'IPPy').exists():
        ippy_root = base / 'IPPy'
        break
else:
    raise FileNotFoundError('Could not locate the local IPPy package.')

operators_spec = importlib.util.spec_from_file_location('course_ippy_operators', ippy_root / 'operators.py')
operators = importlib.util.module_from_spec(operators_spec)
operators_spec.loader.exec_module(operators)

weights_dir = book_root / 'weights'
weights_dir.mkdir(exist_ok=True)

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    try:
        if torch.backends.mps.is_available():
            return 'mps'
    except AttributeError:
        pass
    return 'cpu'

def gaussian_noise(y, noise_level):
    e = torch.randn_like(y, device=y.device)
    return e / torch.norm(e) * torch.norm(y) * noise_level

class MayoDataset(Dataset):
    def __init__(self, data_path, data_shape):
        super().__init__()
        self.data_path = data_path
        self.data_shape = data_shape
        self.fname_list = glob.glob(f'{data_path}/*/*.png')

    def __len__(self):
        return len(self.fname_list)

    def __getitem__(self, idx):
        img_path = self.fname_list[idx]
        x = Image.open(img_path).convert('L')
        x = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(self.data_shape),
        ])(x)
        return x

from torch.utils.data import Subset, random_split

device = get_device()
train_dataset = MayoDataset(data_path=str(book_root / 'Mayo' / 'train'), data_shape=256)
test_dataset = MayoDataset(data_path=str(book_root / 'Mayo' / 'test'), data_shape=256)

# Select a subset of the original set (1/4)
train_subset = Subset(train_dataset, range(len(train_dataset) // 4))
test_subset = Subset(test_dataset, range(len(test_dataset) // 4))

# Crea un validation set dal training set (es. 20%)
val_size = int(0.2 * len(train_subset))
train_size = len(train_subset) - val_size
train_subset, val_subset = random_split(train_subset, [train_size, val_size])

K = operators.Blurring(
    img_shape=(256, 256),
    kernel_type='motion',
    kernel_size=9,
    motion_angle=20,
)

class BlurredDatasetWrapper(Dataset):
    def __init__(self, dataset, blur_op, noise_level=0.01):
        self.dataset = dataset
        self.blur_op = blur_op
        self.noise_level = noise_level

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x = self.dataset[idx]
        x_batched = x.unsqueeze(0) # [1, 1, 256, 256]
        with torch.no_grad():
            y_batched = self.blur_op(x_batched)
            e = torch.randn_like(y_batched)
            y_batched = y_batched + e / torch.norm(e) * torch.norm(y_batched) * self.noise_level
        
        y = y_batched.squeeze(0)
        return y, x

train_blurred_dataset = BlurredDatasetWrapper(train_subset, K)
val_blurred_dataset = BlurredDatasetWrapper(val_subset, K)
test_blurred_dataset = BlurredDatasetWrapper(test_subset, K)

train_loader = DataLoader(train_blurred_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_blurred_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_blurred_dataset, batch_size=8, shuffle=False)

print('Device:', device)
print('Training images:', len(train_blurred_dataset))
print('Validation images:', len(val_blurred_dataset))
print('Test images:', len(test_blurred_dataset))
print('Weights directory:', weights_dir)


Device: mps
Training images: 661
Validation images: 165
Test images: 81
Weights directory: /Users/mattia.dispigno/University/UniBo/computational imaging/Computational Imaging/weights


## Task 2: Define two reconstruction models

Implement two models in **PyTorch**.

1.  **Model A** should only use local convolutions and pointwise nonlinearities.
2.  **Model B** should include one mechanism that enlarges the receptive field, such as pooling or strided convolution, followed by an upsampling step.
3.  Keep both models small enough that training runs quickly.

In [2]:
# Implement Task 2 here

## Task 3: Train both models

Train the two networks on the same data.

1.  Use the same loss function for both models, for example MSE.
2.  Use the same optimizer and roughly the same number of training iterations.
3.  Record the training loss, and if possible also the validation loss.

In [3]:
# Implement Task 3 here

## Task 4: Compare the outputs

Pick one or two validation examples and display:

1.  the clean image,
2.  the blurred input,
3.  the reconstruction produced by Model A,
4.  the reconstruction produced by Model B.

Then answer the following questions in a short paragraph:

*   Which model reconstructs large blurred structures more effectively?
*   What role does the receptive field appear to play?
*   What is the possible drawback of pooling in an imaging problem where fine spatial detail matters?

In [4]:
# Implement visualization and comparison here

## Deliverable

Submit the following four items:

1.  the code defining the two models;
2.  one figure showing the training-loss curves;
3.  one figure comparing the reconstructions on at least one validation example;
4.  a short written discussion of about 8–10 lines explaining what you observed.

A complete deliverable should make it possible to verify both that the models were implemented correctly and that you understood the connection between receptive field, pooling, and image reconstruction.